# Script 14 v4.1 — Expanded Model Selection with Target-Aligned Volatility Scoring

**Purpose:** compare available point-forecast pipelines on matching origins, and score volatility candidates against EWMA on paired origins with explicit target-date mapping. Development targets on or after 2024-01-01 are excluded. This notebook does not use the locked 2024–2025 holdout for selection.

**Important:** Volatility scores include extreme-value/loss-concentration flags. A quality-gate PASS validates the assembled comparison and date boundary, not every upstream model's training process or the statistical superiority of any candidate.


In [ ]:

from pathlib import Path
import math, warnings
from IPython.display import display
import numpy as np
import pandas as pd

HOLDOUT_START = pd.Timestamp("2024-01-01")
RANDOM_SEED = 20261025
N_BOOT = 400
ALPHA = 0.05

# Find the Research project root without hard-coding the user's Windows path.
def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        if (p / "outputs" / "predictions").is_dir():
            return p
    # Helpful fallback for running a copy against a flat folder of supplied CSVs.
    if all((Path.cwd() / name).exists() for name in [
        "04_random_walk_predictions_2019_2023.csv",
        "05_statistical_predictions_2019_2023.csv"]):
        return Path.cwd()
    raise FileNotFoundError(
        "Could not find outputs/predictions. Open this notebook from your Research project, "
        "or place it in Research/notebooks/ and run it again."
    )

PROJECT_ROOT = find_project_root()
PRED_DIR = PROJECT_ROOT / "outputs" / "predictions"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
# Flat-folder fallback is only for test runs; on the user's project, outputs/predictions is used.
if not PRED_DIR.exists():
    PRED_DIR = PROJECT_ROOT
TABLE_DIR.mkdir(parents=True, exist_ok=True)

KEYS = ["Currency", "Horizon", "Forecast_Origin", "Target_Date"]
BASELINE_FILE = "04_random_walk_predictions_2019_2023.csv"

# Candidate files and final prediction columns. These are pipeline-level candidates, not claims
# about every individual algorithm nested inside a selection pipeline.
CANDIDATE_SPECS = [
    ("Script 05 — Statistical pipeline", "05_statistical_predictions_2019_2023.csv", "Stat_Prediction", "all horizons"),
    ("Script 06 — ML pipeline", "06_FAST_ml_predictions_2019_2023.csv", "ML_Prediction", "all horizons"),
    ("Script 07 — Hybrid pipeline", "07_hybrid_predictions_2019_2023.csv", "Hybrid_Prediction", "all horizons"),
    ("Script 08 — Adaptive ensemble", "08_adaptive_ensemble_predictions_2019_2023.csv", "Ensemble_Prediction", "all horizons"),
    ("Script 09 — Online hybrid", "09_online_hybrid_predictions_2019_2023.csv", "Online_Hybrid_Prediction", "all horizons"),
    ("Script 10 — Dedicated H3 final", "10_H3_predictions_2019_2023.csv", "Final_H3_Prediction", "H3 only"),
    ("Script 10B — H3 MI final", "10B_H3_MI_predictions_2019_2023.csv", "Final_Prediction", "H3 only"),
    ("Script 11 — Raw DL component", "11_deep_learning_predictions_2019_2023.csv", "DL_Prediction", "all horizons"),
    ("Script 11 — Final DL/RW hybrid", "11_deep_learning_predictions_2019_2023.csv", "Final_DL_Hybrid_Prediction", "all horizons"),
    ("Script 11B — Regime-gated DL hybrid", "11B_regime_gated_DL_predictions_2019_2023.csv", "Final_Prediction", "all horizons"),
]

# Utilities

def read_csv_required(path):
    if not path.exists():
        raise FileNotFoundError(f"Required prediction file is missing: {path}")
    return pd.read_csv(path, low_memory=False)

def standardize_keys(df, source_name):
    missing = [c for c in KEYS if c not in df.columns]
    if missing:
        raise ValueError(f"{source_name}: missing required key columns {missing}")
    out = df.copy()
    out["Currency"] = out["Currency"].astype(str).str.strip().str.upper()
    out["Horizon"] = pd.to_numeric(out["Horizon"], errors="coerce")
    out["Forecast_Origin"] = pd.to_datetime(out["Forecast_Origin"], errors="coerce")
    out["Target_Date"] = pd.to_datetime(out["Target_Date"], errors="coerce")
    out = out.dropna(subset=KEYS).copy()
    out["Horizon"] = out["Horizon"].astype(int)
    return out

def max_abs_diff(a, b):
    a = pd.to_numeric(pd.Series(a), errors="coerce").to_numpy(dtype=float)
    b = pd.to_numeric(pd.Series(b), errors="coerce").to_numpy(dtype=float)
    ok = np.isfinite(a) & np.isfinite(b)
    return float(np.max(np.abs(a[ok] - b[ok]))) if ok.any() else np.nan

def rmse(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    ok = np.isfinite(y) & np.isfinite(p)
    return float(np.sqrt(np.mean((y[ok] - p[ok])**2))) if ok.any() else np.nan

def mae(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    ok = np.isfinite(y) & np.isfinite(p)
    return float(np.mean(np.abs(y[ok] - p[ok]))) if ok.any() else np.nan

def directional_accuracy(y, p, rw):
    # Direction is measured relative to the forecast-origin value approximated by the RW prediction.
    y, p, rw = np.asarray(y, float), np.asarray(p, float), np.asarray(rw, float)
    ok = np.isfinite(y) & np.isfinite(p) & np.isfinite(rw)
    if not ok.any(): return np.nan
    actual_dir = np.sign(y[ok] - rw[ok])
    pred_dir = np.sign(p[ok] - rw[ok])
    return float(np.mean(actual_dir == pred_dir) * 100)

def holm_adjust(pvalues):
    """Holm family-wise adjusted p-values; NaNs remain NaN."""
    p = np.asarray(pvalues, dtype=float)
    out = np.full(p.shape, np.nan, dtype=float)
    valid = np.where(np.isfinite(p))[0]
    if len(valid) == 0: return out
    order = valid[np.argsort(p[valid])]
    m = len(order)
    adjusted = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        val = min(1.0, (m-rank) * p[idx])
        running = max(running, val)
        adjusted[rank] = running
    for rank, idx in enumerate(order): out[idx] = adjusted[rank]
    return out

def dm_test_candidate_better(candidate_error, rw_error, horizon):
    """Two-sided DM-style test on squared-error loss difference, HAC lag = horizon-1.
    d = candidate squared error - RW squared error; negative mean favours candidate.
    Includes the Harvey-Leybourne-Newbold small-sample adjustment. Returns statistic, p-value, n.
    """
    from scipy.stats import t as student_t
    ec = np.asarray(candidate_error, dtype=float)
    er = np.asarray(rw_error, dtype=float)
    ok = np.isfinite(ec) & np.isfinite(er)
    d = ec[ok]**2 - er[ok]**2
    n = len(d)
    h = max(1, int(horizon))
    if n < max(20, h + 5): return np.nan, np.nan, n
    dbar = float(np.mean(d))
    x = d - dbar
    max_lag = min(h-1, n-1)
    lrv = float(np.dot(x, x) / n)
    for lag in range(1, max_lag+1):
        gamma = float(np.dot(x[lag:], x[:-lag]) / n)
        weight = 1.0 - lag/(max_lag+1.0)
        lrv += 2.0 * weight * gamma
    if not np.isfinite(lrv) or lrv <= 1e-18:
        if abs(dbar) <= 1e-14: return 0.0, 1.0, n
        return np.nan, np.nan, n
    stat = dbar / math.sqrt(lrv/n)
    hln_term = (n + 1 - 2*h + h*(h-1)/n) / n
    if hln_term > 0: stat *= math.sqrt(hln_term)
    p = float(2 * student_t.sf(abs(stat), df=n-1))
    return float(stat), p, n

def moving_block_bootstrap_skill(y, candidate, rw, horizon, n_boot=N_BOOT, seed=RANDOM_SEED):
    """Paired circular moving-block bootstrap CI for RMSE skill (%)."""
    y = np.asarray(y, float); candidate = np.asarray(candidate, float); rw = np.asarray(rw, float)
    ok = np.isfinite(y) & np.isfinite(candidate) & np.isfinite(rw)
    y, candidate, rw = y[ok], candidate[ok], rw[ok]
    n = len(y)
    if n < 50: return np.nan, np.nan
    block = max(1, int(horizon))
    rng = np.random.default_rng(seed + n + block)
    n_blocks = int(np.ceil(n / block))
    offsets = np.arange(block)
    boot_skills = np.empty(n_boot)
    for b in range(n_boot):
        starts = rng.integers(0, n, size=n_blocks)
        idx = ((starts[:, None] + offsets[None, :]) % n).ravel()[:n]
        rmse_c = np.sqrt(np.mean((y[idx] - candidate[idx])**2))
        rmse_r = np.sqrt(np.mean((y[idx] - rw[idx])**2))
        boot_skills[b] = 100*(rmse_r-rmse_c)/rmse_r if rmse_r > 0 else np.nan
    lo, hi = np.nanpercentile(boot_skills, [2.5, 97.5])
    return float(lo), float(hi)

# Deterministic sanity checks for the corrected DM-style calculation.
_rng_test = np.random.default_rng(314159)
_rw_test_error = _rng_test.normal(0, 1, 3000)
_better_test_error = 0.40 * _rw_test_error + _rng_test.normal(0, 0.05, 3000)
_worse_test_error = 1.60 * _rw_test_error + _rng_test.normal(0, 0.05, 3000)
_equal_stat, _equal_p, _ = dm_test_candidate_better(_rw_test_error, _rw_test_error, 1)
_better_stat, _better_p, _ = dm_test_candidate_better(_better_test_error, _rw_test_error, 1)
_worse_stat, _worse_p, _ = dm_test_candidate_better(_worse_test_error, _rw_test_error, 1)
DM_SANITY_PASS = bool(_equal_p == 1.0 and _better_stat < 0 and _better_p < 0.05 and _worse_stat > 0 and _worse_p < 0.05)
print("DM diagnostic sanity test:", "PASS" if DM_SANITY_PASS else "FAIL")
print("  equal series p-value:", round(_equal_p, 6), "| better candidate stat/p:", round(_better_stat, 4), round(_better_p, 6), "| worse candidate stat/p:", round(_worse_stat, 4), round(_worse_p, 6))

print("Project root:", PROJECT_ROOT)
print("Prediction folder:", PRED_DIR)
print("Table output folder:", TABLE_DIR)


In [ ]:

# Load and audit the Random Walk benchmark
rw_raw = standardize_keys(read_csv_required(PRED_DIR / BASELINE_FILE), BASELINE_FILE)
for col in ["Actual", "RW_Prediction"]:
    if col not in rw_raw.columns:
        raise ValueError(f"{BASELINE_FILE}: missing required column {col}")
for col in ["Actual", "RW_Prediction"]:
    rw_raw[col] = pd.to_numeric(rw_raw[col], errors="coerce")

# A unique key is required for paired evaluation. Do not silently choose among conflicting duplicates.
dup_rw = rw_raw.duplicated(KEYS, keep=False)
if dup_rw.any():
    raise ValueError(f"Random Walk file has {int(dup_rw.sum())} rows in duplicate forecast keys. Resolve duplicates first.")

rw_all = rw_raw[KEYS + ["Actual", "RW_Prediction"]].copy()
rw_before = len(rw_all)
rw_crossing = int((rw_all["Target_Date"] >= HOLDOUT_START).sum())
rw_dev = rw_all.loc[rw_all["Target_Date"] < HOLDOUT_START].copy()

candidate_long_parts = []
input_audit_rows = []
integrity_rows = []

for candidate_name, filename, pred_col, scope in CANDIDATE_SPECS:
    path = PRED_DIR / filename
    if not path.exists():
        input_audit_rows.append({"Candidate": candidate_name, "File": filename, "Prediction_Column": pred_col,
                                 "Status": "MISSING", "Rows_Raw": 0, "Rows_Dev": 0, "Rows_Target_Crossing_Holdout": 0,
                                 "Eligible_Tasks": 0, "Evidence": "File not found; candidate not evaluated."})
        continue
    src = standardize_keys(read_csv_required(path), filename)
    if pred_col not in src.columns:
        input_audit_rows.append({"Candidate": candidate_name, "File": filename, "Prediction_Column": pred_col,
                                 "Status": "SKIP", "Rows_Raw": len(src), "Rows_Dev": 0, "Rows_Target_Crossing_Holdout": 0,
                                 "Eligible_Tasks": 0, "Evidence": f"Missing prediction column {pred_col}."})
        continue
    for col in ["Actual", "RW_Prediction", pred_col]:
        if col not in src.columns:
            raise ValueError(f"{filename}: missing required column {col}")
        src[col] = pd.to_numeric(src[col], errors="coerce")
    dup = src.duplicated(KEYS, keep=False)
    if dup.any():
        raise ValueError(f"{filename} has {int(dup.sum())} rows in duplicate forecast keys. Resolve duplicates first.")
    raw_n = len(src)
    cross_n = int((src["Target_Date"] >= HOLDOUT_START).sum())
    dev = src.loc[src["Target_Date"] < HOLDOUT_START].copy()
    # Pair with the canonical RW file and compare actual + RW values for every shared key.
    paired = dev.merge(rw_dev, on=KEYS, how="inner", suffixes=("_src", "_base"), validate="one_to_one")
    missing_rw = len(dev) - len(paired)
    if len(paired):
        actual_diff = max_abs_diff(paired["Actual_src"], paired["Actual_base"])
        rw_diff = max_abs_diff(paired["RW_Prediction_src"], paired["RW_Prediction_base"])
    else:
        actual_diff = rw_diff = np.nan
    integrity_rows.append({"Candidate": candidate_name, "File": filename, "Rows_Dev": len(dev), "Matched_RW_Rows": len(paired),
                           "Rows_Without_RW_Match": missing_rw, "Max_Abs_Actual_Difference": actual_diff,
                           "Max_Abs_RW_Difference": rw_diff,
                           "Integrity_Status": "PASS" if len(paired)>0 and missing_rw==0 and actual_diff < 1e-8 and rw_diff < 1e-8 else "REVIEW"})
    if paired.empty:
        input_audit_rows.append({"Candidate": candidate_name, "File": filename, "Prediction_Column": pred_col,
                                 "Status": "SKIP", "Rows_Raw": raw_n, "Rows_Dev": len(dev), "Rows_Target_Crossing_Holdout": cross_n,
                                 "Eligible_Tasks": 0, "Evidence": "No matching development rows in Random Walk baseline."})
        continue
    # Use canonical actual and RW values after verifying source consistency.
    pred_values = paired[pred_col]
    cand = pd.DataFrame({
        **{k: paired[k].values for k in KEYS},
        "Candidate": candidate_name,
        "Prediction": pred_values.values,
        "Actual": paired["Actual_base"].values,
        "RW_Prediction": paired["RW_Prediction_base"].values,
        "Source_File": filename,
        "Prediction_Column": pred_col,
    })
    cand = cand[np.isfinite(cand["Prediction"]) & np.isfinite(cand["Actual"]) & np.isfinite(cand["RW_Prediction"])].copy()
    candidate_long_parts.append(cand)
    tasks = cand[["Currency", "Horizon"]].drop_duplicates().shape[0]
    status = "PASS" if tasks > 0 and missing_rw == 0 and actual_diff < 1e-8 and rw_diff < 1e-8 else "REVIEW"
    input_audit_rows.append({"Candidate": candidate_name, "File": filename, "Prediction_Column": pred_col,
                             "Status": status, "Rows_Raw": raw_n, "Rows_Dev": len(dev), "Rows_Target_Crossing_Holdout": cross_n,
                             "Eligible_Tasks": tasks,
                             "Evidence": f"Prediction column accepted; matched {len(paired)} rows; actual max diff={actual_diff:.3g}; RW max diff={rw_diff:.3g}."})

# Add the baseline itself as an explicitly reported candidate so rankings can include it.
rw_candidate = rw_dev.copy()
rw_candidate["Candidate"] = "Random Walk benchmark"
rw_candidate["Prediction"] = rw_candidate["RW_Prediction"]
rw_candidate["Source_File"] = BASELINE_FILE
rw_candidate["Prediction_Column"] = "RW_Prediction"
candidate_long_parts.append(rw_candidate[KEYS + ["Candidate", "Prediction", "Actual", "RW_Prediction", "Source_File", "Prediction_Column"]])

candidate_long = pd.concat(candidate_long_parts, ignore_index=True) if candidate_long_parts else pd.DataFrame()
input_audit = pd.DataFrame(input_audit_rows)
origin_integrity = pd.DataFrame(integrity_rows)
point_target_boundary_audit = pd.DataFrame([{
    "Holdout_Start": HOLDOUT_START.date().isoformat(),
    "Baseline_Rows_Before_Filter": rw_before,
    "Baseline_Target_Rows_Excluded": rw_crossing,
    "Baseline_Development_Rows_Retained": len(rw_dev),
    "Latest_Retained_Target_Date": rw_dev["Target_Date"].max().date().isoformat() if len(rw_dev) else None,
    "Boundary_Rule": "Keep only Target_Date < 2024-01-01; origins in 2023 whose targets enter 2024 are excluded."
}])

print("Candidate inputs:")
display(input_audit)
print("Integrity audit:")
display(origin_integrity)
print("Point target-date boundary:")
display(point_target_boundary_audit)


In [ ]:

# Score every candidate against RW for each currency-horizon task.
score_rows = []
if candidate_long.empty:
    raise ValueError("No candidate predictions were loaded. Check the file paths and prediction columns.")

for (candidate, currency, horizon), g in candidate_long.groupby(["Candidate", "Currency", "Horizon"], sort=True):
    g = g.sort_values(["Forecast_Origin", "Target_Date"])
    y = g["Actual"].to_numpy(float)
    p = g["Prediction"].to_numpy(float)
    rw = g["RW_Prediction"].to_numpy(float)
    n = len(g)
    r_c = rmse(y, p); r_rw = rmse(y, rw)
    skill = 100*(r_rw-r_c)/r_rw if np.isfinite(r_c) and np.isfinite(r_rw) and r_rw > 0 else np.nan
    m_c = mae(y, p); m_rw = mae(y, rw)
    err_c = y-p; err_rw = y-rw
    if candidate == "Random Walk benchmark":
        dm_stat, dm_p, dm_n = 0.0, 1.0, n
        ci_lo, ci_hi = 0.0, 0.0
    else:
        dm_stat, dm_p, dm_n = dm_test_candidate_better(err_c, err_rw, int(horizon))
        ci_lo, ci_hi = moving_block_bootstrap_skill(y, p, rw, int(horizon))
    score_rows.append({
        "Candidate": candidate, "Currency": currency, "Horizon": int(horizon), "N": n,
        "RMSE": r_c, "MAE": m_c, "RW_RMSE": r_rw, "RW_MAE": m_rw,
        "RMSE_Skill_vs_RW_percent": skill,
        "MAE_Skill_vs_RW_percent": 100*(m_rw-m_c)/m_rw if m_rw>0 else np.nan,
        "Directional_Accuracy_percent": directional_accuracy(y, p, rw),
        "DM_Statistic": dm_stat, "DM_P_Raw_TwoSided": dm_p, "DM_N": dm_n,
        "Bootstrap_Skill_CI_Lower": ci_lo, "Bootstrap_Skill_CI_Upper": ci_hi,
        "Target_Start": g["Target_Date"].min().date().isoformat(),
        "Target_End": g["Target_Date"].max().date().isoformat(),
        "Source_File": g["Source_File"].iloc[0], "Prediction_Column": g["Prediction_Column"].iloc[0]
    })

point_task_scores = pd.DataFrame(score_rows)
# Holm adjustment in two scopes: across every candidate-task test, and within each currency-horizon task.
mask_non_rw = point_task_scores["Candidate"] != "Random Walk benchmark"
point_task_scores["DM_P_Holm_Global"] = np.nan
point_task_scores.loc[~mask_non_rw, "DM_P_Holm_Global"] = 1.0
point_task_scores.loc[mask_non_rw, "DM_P_Holm_Global"] = holm_adjust(point_task_scores.loc[mask_non_rw, "DM_P_Raw_TwoSided"].to_numpy())
point_task_scores["DM_P_Holm_Within_Task"] = np.nan
for task_key, idx in point_task_scores.groupby(["Currency", "Horizon"]).groups.items():
    idx = list(idx)
    sub_idx = [i for i in idx if point_task_scores.loc[i, "Candidate"] != "Random Walk benchmark"]
    if sub_idx:
        adj = holm_adjust(point_task_scores.loc[sub_idx, "DM_P_Raw_TwoSided"].to_numpy())
        point_task_scores.loc[sub_idx, "DM_P_Holm_Within_Task"] = adj
    rw_idx = [i for i in idx if point_task_scores.loc[i, "Candidate"] == "Random Walk benchmark"]
    if rw_idx: point_task_scores.loc[rw_idx, "DM_P_Holm_Within_Task"] = 1.0

# Candidate-level descriptive summary. H3-only candidates have only four eligible tasks.
global_rows = []
for candidate, g in point_task_scores.groupby("Candidate", sort=False):
    non_rw = candidate != "Random Walk benchmark"
    global_rows.append({
        "Candidate": candidate,
        "Eligible_Tasks": int(g.shape[0]),
        "Tasks_Lower_RMSE_than_RW": int((g["RMSE_Skill_vs_RW_percent"] > 0).sum()),
        "Tasks_Equal_RMSE_to_RW": int(np.isclose(g["RMSE_Skill_vs_RW_percent"], 0, atol=1e-12).sum()),
        "Tasks_Higher_RMSE_than_RW": int((g["RMSE_Skill_vs_RW_percent"] < 0).sum()),
        "Mean_Task_RMSE_Skill_percent": float(g["RMSE_Skill_vs_RW_percent"].mean()),
        "Median_Task_RMSE_Skill_percent": float(g["RMSE_Skill_vs_RW_percent"].median()),
        "Mean_RMSE": float(g["RMSE"].mean()),
        "Mean_MAE": float(g["MAE"].mean()),
        "Tasks_Bootstrap_CI_Lower_Above_Zero": int((g["Bootstrap_Skill_CI_Lower"] > 0).sum()) if non_rw else 0,
        "Tasks_DM_Holm_Global_lt_0_05_Any_Difference": int((g["DM_P_Holm_Global"] < ALPHA).sum()) if non_rw else 0,
        "Tasks_DM_Holm_Global_lt_0_05_and_Better": int(((g["DM_P_Holm_Global"] < ALPHA) & (g["RMSE_Skill_vs_RW_percent"] > 0)).sum()) if non_rw else 0,
        "Tasks_DM_Holm_Global_lt_0_05_and_Worse": int(((g["DM_P_Holm_Global"] < ALPHA) & (g["RMSE_Skill_vs_RW_percent"] < 0)).sum()) if non_rw else 0,
        "Scope_Note": "H3 only; compare only with H3 candidates" if candidate.startswith("Script 10") else ("Benchmark" if not non_rw else "All available horizons; pipeline-level candidate")
    })
point_global_scores = pd.DataFrame(global_rows).sort_values("Mean_Task_RMSE_Skill_percent", ascending=False)

# For each task, report the best observed candidate (including RW) separately from robust evidence.
task_decision_rows = []
for (currency, horizon), g in point_task_scores.groupby(["Currency", "Horizon"], sort=True):
    g = g.sort_values(["RMSE", "Candidate"])
    best = g.iloc[0]
    rw_row = g[g["Candidate"] == "Random Walk benchmark"].iloc[0]
    if best["Candidate"] == "Random Walk benchmark":
        best_label = "Random Walk benchmark"
        best_status = "RW has the lowest observed RMSE"
    else:
        best_label = best["Candidate"]
        supported = (float(best["Bootstrap_Skill_CI_Lower"]) > 0 and
                     float(best["DM_P_Holm_Global"]) < ALPHA and
                     float(best["RMSE_Skill_vs_RW_percent"]) > 0)
        best_status = "Robust improvement supported by current criteria" if supported else "Lowest observed RMSE; superiority not statistically established"
    # Conservative selection only replaces RW if a candidate has robust positive skill.
    task_candidates = g[(g["Candidate"] != "Random Walk benchmark") &
                        (g["RMSE_Skill_vs_RW_percent"] > 0) &
                        (g["Bootstrap_Skill_CI_Lower"] > 0) &
                        (g["DM_P_Holm_Global"] < ALPHA)]
    if len(task_candidates):
        selected = task_candidates.sort_values("RMSE").iloc[0]
        conservative = selected["Candidate"]
        conservative_status = "Robust candidate passed global Holm + bootstrap CI rule"
    else:
        conservative = "Random Walk benchmark"
        conservative_status = "Conservative default; no candidate passed both global evidence rules"
    task_decision_rows.append({
        "Currency": currency, "Horizon": int(horizon), "Best_Observed_Candidate": best_label,
        "Best_Observed_RMSE": float(best["RMSE"]), "Best_Observed_RMSE_Skill_vs_RW_percent": float(best["RMSE_Skill_vs_RW_percent"]),
        "Best_Observed_Status": best_status,
        "Random_Walk_RMSE": float(rw_row["RMSE"]),
        "Conservative_Statistical_Selection": conservative,
        "Conservative_Selection_Status": conservative_status,
        "Eligible_Candidate_Count_Including_RW": int(len(g))
    })
point_task_decisions = pd.DataFrame(task_decision_rows)

# Coverage summary by candidate and task.
point_candidate_coverage = (point_task_scores.groupby("Candidate", as_index=False)
    .agg(Eligible_Tasks=("Horizon", "size"),
         Total_Predictions=("N", "sum"),
         Min_N=("N", "min"), Max_N=("N", "max")))

# Display useful tables.
print("GLOBAL CANDIDATE SUMMARY — descriptive skill; not a guarantee of statistical superiority")
display(point_global_scores.round(4))
print("TASK-BY-TASK DECISIONS — observed winner and conservative evidence-based decision are separate")
display(point_task_decisions.round(5))


In [ ]:
# Separate volatility comparison against EWMA using matched forecast origins and explicit target dates.
# Target_Date is taken from the same currency/horizon/origin map as the point-forecast benchmark.
# Rows whose target date is on/after 2024-01-01 are excluded from development-only selection.

VOL_HOLDOUT_START = pd.Timestamp("2024-01-01")
VOL_KEY = ["Currency", "Horizon", "Forecast_Origin"]

vol_target_map = rw_raw[["Currency", "Horizon", "Forecast_Origin", "Target_Date"]].drop_duplicates().copy()
if vol_target_map.duplicated(VOL_KEY).any():
    raise ValueError("STOP: Random Walk file has duplicate currency/horizon/origin rows for volatility target mapping.")

vol_target_map["Target_Date"] = pd.to_datetime(vol_target_map["Target_Date"], errors="coerce")
vol_target_map = vol_target_map.dropna(subset=VOL_KEY + ["Target_Date"])

def qlike_loss(realized, forecast):
    realized = np.asarray(realized, float)
    forecast = np.asarray(forecast, float)
    forecast = np.maximum(forecast, 1e-12)
    return np.log(forecast) + realized / forecast

def score_vol_pair(candidate_name, frame, realized_col, forecast_col, benchmark_col, source_name):
    d = frame.copy()
    required = VOL_KEY + ["Target_Date", realized_col, forecast_col, benchmark_col]
    missing = [c for c in required if c not in d.columns]
    if missing:
        vol_audit_rows.append({
            "Candidate": candidate_name, "File": source_name, "Status": "SKIP",
            "Rows_Before_Filter": len(d), "Rows_Scored": 0,
            "Target_Date_Available": False, "Rows_Target_Map_Missing": np.nan,
            "Rows_Excluded_Holdout_Target": np.nan,
            "Evidence": f"Missing required columns: {missing}"
        })
        return

    for c in [realized_col, forecast_col, benchmark_col]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.dropna(subset=VOL_KEY + ["Target_Date", realized_col, forecast_col, benchmark_col]).copy()
    d = d[
        np.isfinite(d[realized_col]) & np.isfinite(d[forecast_col]) & np.isfinite(d[benchmark_col])
        & (d[realized_col] >= 0) & (d[forecast_col] > 0) & (d[benchmark_col] > 0)
    ].copy()

    before_boundary = len(d)
    excluded_holdout = int((d["Target_Date"] >= VOL_HOLDOUT_START).sum())
    d = d[d["Target_Date"] < VOL_HOLDOUT_START].copy()
    d = d[d["Forecast_Origin"].dt.year.between(2019, 2023)].copy()

    # The candidate and EWMA values in each row are paired on the exact same origin.
    vol_audit_rows.append({
        "Candidate": candidate_name, "File": source_name, "Status": "PASS" if len(d) else "SKIP",
        "Rows_Before_Filter": int(len(frame)), "Rows_Scored": int(len(d)),
        "Target_Date_Available": True, "Rows_Target_Map_Missing": 0,
        "Rows_Excluded_Holdout_Target": excluded_holdout,
        "Latest_Target_Date_Scored": str(d["Target_Date"].max().date()) if len(d) else "",
        "Evidence": "Candidate and EWMA scored on paired origins; targets on/after 2024-01-01 excluded."
    })

    for (currency, horizon), g in d.groupby(["Currency", "Horizon"], sort=True):
        y = g[realized_col].to_numpy(float)
        f = g[forecast_col].to_numpy(float)
        b = g[benchmark_col].to_numpy(float)
        with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
            qf = qlike_loss(y, f)
            qb = qlike_loss(y, b)
            mse_f = float(np.mean((y - f) ** 2))
            mse_b = float(np.mean((y - b) ** 2))
        per_origin_gain = qb - qf
        abs_gain = np.abs(per_origin_gain[np.isfinite(per_origin_gain)])
        if len(abs_gain):
            top_k = max(1, int(np.ceil(0.01 * len(abs_gain))))
            top_share = float(np.sort(abs_gain)[-top_k:].sum() / max(abs_gain.sum(), 1e-12))
            max_abs_gain = float(abs_gain.max())
        else:
            top_share = np.nan
            max_abs_gain = np.nan
        extreme_forecast_rows = int(((f < 1e-8) | (f > 1e6)).sum())
        extreme_loss_flag = bool(
            extreme_forecast_rows > 0
            or not np.isfinite(mse_f)
            or (np.isfinite(top_share) and top_share > 0.50)
            or not np.isfinite(np.mean(qf))
        )
        vol_rows.append({
            "Candidate": candidate_name,
            "Currency": currency,
            "Horizon": int(horizon),
            "N": int(len(g)),
            "First_Forecast_Origin": str(g["Forecast_Origin"].min().date()),
            "Last_Forecast_Origin": str(g["Forecast_Origin"].max().date()),
            "Latest_Target_Date": str(g["Target_Date"].max().date()),
            "QLIKE_Candidate": float(np.mean(qf)),
            "QLIKE_EWMA": float(np.mean(qb)),
            "QLIKE_Gain_vs_EWMA": float(np.mean(qb) - np.mean(qf)),
            "Variance_MSE_Candidate": mse_f,
            "Variance_MSE_EWMA": mse_b,
            "Variance_MSE_Skill_vs_EWMA_percent": 100 * (mse_b - mse_f) / mse_b if mse_b > 0 else np.nan,
            "Extreme_Forecast_Rows": extreme_forecast_rows,
            "Top_1pct_Absolute_QLIKE_Difference_Share": top_share,
            "Max_Absolute_QLIKE_Difference": max_abs_gain,
            "Extreme_Loss_Concentration_Flag": extreme_loss_flag,
            "Status": "REVIEW_EXTREME_VALUES_OR_LOSS_CONCENTRATION" if extreme_loss_flag else "DEVELOPMENT_TARGET_BOUNDARY_VERIFIED"
        })

vol_rows = []
vol_audit_rows = []

# 1) Individual volatility models from the long-format Script 12C candidate file.
# This source includes one row per model/origin and an EWMA94 row for a proper paired benchmark.
long_name = "12C_corrected_candidate_predictions_2018_2023.csv"
long_path = PRED_DIR / long_name
if long_path.exists():
    long_df = pd.read_csv(long_path, low_memory=False)
    needed = ["Currency", "Horizon", "Forecast_Origin", "Realized_Variance", "Model", "Variance_Forecast"]
    missing = [c for c in needed if c not in long_df.columns]
    if missing:
        vol_audit_rows.append({
            "Candidate": "Script 12C individual volatility models", "File": long_name, "Status": "SKIP",
            "Rows_Before_Filter": len(long_df), "Rows_Scored": 0, "Target_Date_Available": False,
            "Rows_Target_Map_Missing": np.nan, "Rows_Excluded_Holdout_Target": np.nan,
            "Evidence": f"Missing required columns: {missing}"
        })
    else:
        long_df["Currency"] = long_df["Currency"].astype(str).str.strip().str.upper()
        long_df["Horizon"] = pd.to_numeric(long_df["Horizon"], errors="coerce")
        long_df["Forecast_Origin"] = pd.to_datetime(long_df["Forecast_Origin"], errors="coerce")
        long_df["Realized_Variance"] = pd.to_numeric(long_df["Realized_Variance"], errors="coerce")
        long_df["Variance_Forecast"] = pd.to_numeric(long_df["Variance_Forecast"], errors="coerce")
        long_df = long_df.dropna(subset=VOL_KEY + ["Realized_Variance", "Model", "Variance_Forecast"]).copy()
        long_df = long_df[long_df["Forecast_Origin"].dt.year.between(2019, 2023)].copy()
        long_df = long_df.merge(vol_target_map, on=VOL_KEY, how="left", validate="many_to_one")
        missing_target_rows = int(long_df["Target_Date"].isna().sum())
        long_df = long_df.dropna(subset=["Target_Date"]).copy()

        baseline = long_df[long_df["Model"].astype(str).str.upper().eq("EWMA94")].copy()
        baseline = baseline[["Currency", "Horizon", "Forecast_Origin", "Target_Date", "Realized_Variance", "Variance_Forecast"]]
        baseline = baseline.rename(columns={
            "Realized_Variance": "Realized_Variance_EWMA",
            "Variance_Forecast": "EWMA94_Benchmark"
        })
        if baseline.duplicated(VOL_KEY).any():
            raise ValueError("STOP: duplicate EWMA94 rows in long-format volatility candidate file.")

        model_names = sorted(
            m for m in long_df["Model"].astype(str).unique()
            if m.strip().upper() != "EWMA94"
        )
        for model_name in model_names:
            candidate_df = long_df[long_df["Model"].astype(str).eq(model_name)].copy()
            candidate_df = candidate_df[[
                "Currency", "Horizon", "Forecast_Origin", "Target_Date",
                "Realized_Variance", "Variance_Forecast"
            ]].rename(columns={
                "Realized_Variance": "Realized_Variance_Candidate",
                "Variance_Forecast": "Candidate_Forecast"
            })
            paired = candidate_df.merge(
                baseline,
                on=VOL_KEY + ["Target_Date"],
                how="inner",
                validate="one_to_one"
            )
            if len(paired):
                rv_diff = np.abs(
                    paired["Realized_Variance_Candidate"].to_numpy(float)
                    - paired["Realized_Variance_EWMA"].to_numpy(float)
                )
                if np.nanmax(rv_diff) > 1e-8:
                    raise ValueError(
                        f"STOP: realized-variance target mismatch between {model_name} and EWMA94 "
                        f"(max absolute difference {np.nanmax(rv_diff):.3g})."
                    )
            paired["Realized_Variance"] = paired["Realized_Variance_Candidate"]
            score_vol_pair(
                f"Script 12C — {model_name}",
                paired,
                "Realized_Variance", "Candidate_Forecast", "EWMA94_Benchmark",
                long_name
            )
        vol_audit_rows.append({
            "Candidate": "Script 12C long-format target map",
            "File": long_name,
            "Status": "PASS" if missing_target_rows == 0 else "REVIEW",
            "Rows_Before_Filter": int(len(long_df)),
            "Rows_Scored": int(len(long_df)),
            "Target_Date_Available": bool(missing_target_rows == 0),
            "Rows_Target_Map_Missing": missing_target_rows,
            "Rows_Excluded_Holdout_Target": int((long_df["Target_Date"] >= VOL_HOLDOUT_START).sum()),
            "Latest_Target_Date_Scored": str(long_df.loc[long_df["Target_Date"] < VOL_HOLDOUT_START, "Target_Date"].max().date())
                if (long_df["Target_Date"] < VOL_HOLDOUT_START).any() else "",
            "Evidence": "Forecast origins mapped to Target_Date using the verified point-forecast origin/horizon map."
        })
else:
    vol_audit_rows.append({
        "Candidate": "Script 12C individual volatility models", "File": long_name, "Status": "MISSING",
        "Rows_Before_Filter": 0, "Rows_Scored": 0, "Target_Date_Available": False,
        "Rows_Target_Map_Missing": np.nan, "Rows_Excluded_Holdout_Target": np.nan,
        "Evidence": "Long-format Script 12C candidate file not found."
    })

# 2) Saved Script 12 blend, Script 12B regime-robust, and Script 12C corrected strategies.
wide_specs = [
    ("Script 12 — Blend forecast", "12_volatility_final_predictions_2019_2023.csv",
     "Realized_Variance", "Blend_Forecast", "EWMA94_Forecast"),
    ("Script 12B — Regime-robust forecast", "12B_regime_robust_volatility_predictions_2019_2023.csv",
     "Realized_Variance", "Regime_Robust_Forecast", "EWMA_Benchmark"),
    ("Script 12C — Corrected regime-robust forecast", "12C_corrected_regime_robust_volatility_predictions_2019_2023.csv",
     "Realized_Variance", "Regime_Robust_Forecast", "EWMA_Benchmark"),
]
for candidate_name, filename, rv_col, forecast_col, benchmark_col in wide_specs:
    path = PRED_DIR / filename
    if not path.exists():
        vol_audit_rows.append({
            "Candidate": candidate_name, "File": filename, "Status": "MISSING",
            "Rows_Before_Filter": 0, "Rows_Scored": 0, "Target_Date_Available": False,
            "Rows_Target_Map_Missing": np.nan, "Rows_Excluded_Holdout_Target": np.nan,
            "Evidence": "File not found."
        })
        continue
    d = pd.read_csv(path, low_memory=False)
    required = VOL_KEY + [rv_col, forecast_col, benchmark_col]
    missing = [c for c in required if c not in d.columns]
    if missing:
        vol_audit_rows.append({
            "Candidate": candidate_name, "File": filename, "Status": "SKIP",
            "Rows_Before_Filter": len(d), "Rows_Scored": 0, "Target_Date_Available": False,
            "Rows_Target_Map_Missing": np.nan, "Rows_Excluded_Holdout_Target": np.nan,
            "Evidence": f"Missing required columns: {missing}"
        })
        continue
    d["Currency"] = d["Currency"].astype(str).str.strip().str.upper()
    d["Horizon"] = pd.to_numeric(d["Horizon"], errors="coerce")
    d["Forecast_Origin"] = pd.to_datetime(d["Forecast_Origin"], errors="coerce")
    d = d.dropna(subset=VOL_KEY).copy()
    d = d[d["Forecast_Origin"].dt.year.between(2019, 2023)].copy()
    d = d.merge(vol_target_map, on=VOL_KEY, how="left", validate="many_to_one")
    target_map_missing = int(d["Target_Date"].isna().sum())
    d = d.dropna(subset=["Target_Date"]).copy()
    score_vol_pair(candidate_name, d, rv_col, forecast_col, benchmark_col, filename)
    # Replace the generic audit row from score_vol_pair with explicit map-coverage information.
    if vol_audit_rows and vol_audit_rows[-1].get("Candidate") == candidate_name:
        vol_audit_rows[-1]["Rows_Target_Map_Missing"] = target_map_missing
        if target_map_missing:
            vol_audit_rows[-1]["Status"] = "REVIEW"
            vol_audit_rows[-1]["Evidence"] = (
                f"{target_map_missing} rows had no matching target-date map; remaining rows scored "
                "on paired candidate/EWMA origins."
            )

volatility_task_scores = pd.DataFrame(vol_rows)
volatility_input_audit = pd.DataFrame(vol_audit_rows)

if not volatility_task_scores.empty:
    volatility_global_scores = (
        volatility_task_scores.groupby("Candidate", as_index=False)
        .agg(
            Eligible_Tasks=("Horizon", "size"),
            Total_Observations=("N", "sum"),
            Mean_QLIKE_Gain_vs_EWMA=("QLIKE_Gain_vs_EWMA", "mean"),
            Tasks_Lower_QLIKE_than_EWMA=("QLIKE_Gain_vs_EWMA", lambda x: int((x > 0).sum())),
            Mean_Variance_MSE_Skill_percent=("Variance_MSE_Skill_vs_EWMA_percent", "mean"),
            Tasks_Flagged_Extreme=("Extreme_Loss_Concentration_Flag", "sum")
        )
    )
    print("VOLATILITY COMPARISON — paired candidate/EWMA origins; target dates on/after 2024-01-01 excluded")
    display(volatility_global_scores.round(5))
    display(volatility_input_audit)
else:
    volatility_global_scores = pd.DataFrame()
    print("No volatility candidate scores were created. Check the volatility input audit.")

In [ ]:

# Save all results with a v4 prefix so previous Script 14 outputs are not overwritten.
outputs = {
    "14_v4_1_point_input_audit.csv": input_audit,
    "14_v4_1_point_origin_integrity.csv": origin_integrity,
    "14_v4_1_point_target_boundary_audit.csv": point_target_boundary_audit,
    "14_v4_1_point_task_scores.csv": point_task_scores,
    "14_v4_1_point_global_scores.csv": point_global_scores,
    "14_v4_1_point_task_decisions.csv": point_task_decisions,
    "14_v4_1_point_candidate_coverage.csv": point_candidate_coverage,
    "14_v4_1_volatility_input_audit.csv": volatility_input_audit,
    "14_v4_1_volatility_task_scores.csv": volatility_task_scores,
    "14_v4_1_volatility_global_scores.csv": volatility_global_scores,
    "14_v4_1_volatility_target_boundary_audit.csv": volatility_input_audit,
}
for name, frame in outputs.items():
    frame.to_csv(TABLE_DIR / name, index=False)
    print("Saved:", TABLE_DIR / name)

# Basic guardrails. These checks do not prove upstream training is leakage-free; they check the
# assembled comparison and the target-date boundary represented by the prediction files.
checks = []
def add_check(name, passed, detail): checks.append({"Check":name,"Status":"PASS" if passed else "FAIL","Detail":detail})
add_check("DM diagnostic sanity test", DM_SANITY_PASS, "Synthetic better/equal/worse cases have expected statistic directions and p-values")
add_check("All configured point candidate inputs loaded", bool((input_audit["Status"] == "PASS").all()) and len(input_audit)==len(CANDIDATE_SPECS), f"{int((input_audit['Status']=='PASS').sum())}/{len(CANDIDATE_SPECS)} candidate series loaded")
add_check("RW baseline loaded", len(rw_dev)>0, f"{len(rw_dev)} development rows retained")
add_check("Latest point target before holdout", bool(rw_dev["Target_Date"].max() < HOLDOUT_START), str(rw_dev["Target_Date"].max()))
add_check("No point targets in holdout", bool((candidate_long["Target_Date"] >= HOLDOUT_START).sum()==0), f"{int((candidate_long['Target_Date'] >= HOLDOUT_START).sum())} candidate rows target 2024 or later")
add_check("Point source integrity", bool((origin_integrity["Integrity_Status"]=="PASS").all()) if len(origin_integrity) else False,
          f"{int((origin_integrity['Integrity_Status']=='PASS').sum())}/{len(origin_integrity)} loaded sources passed actual/RW matching")
add_check("At least one non-RW candidate evaluated", bool((point_task_scores["Candidate"]!="Random Walk benchmark").any()), f"{point_task_scores['Candidate'].nunique()-1} non-RW candidate series")
add_check("Task decision table covers 12 tasks", point_task_decisions.shape[0]==12, f"{point_task_decisions.shape[0]} tasks")
add_check("Volatility target-date audit available", bool(len(volatility_input_audit) > 0 and volatility_input_audit["Target_Date_Available"].fillna(False).astype(bool).all() and (volatility_input_audit["Status"] == "PASS").all()), "All volatility source rows must map to point-forecast target dates")
add_check("Volatility target boundary respected", bool(len(volatility_task_scores) > 0 and volatility_task_scores["Latest_Target_Date"].notna().all() and (pd.to_datetime(volatility_task_scores["Latest_Target_Date"], errors="coerce") < HOLDOUT_START).all()), "All scored volatility tasks end before 2024-01-01")
quality_gate = pd.DataFrame(checks)
quality_gate.to_csv(TABLE_DIR / "14_v4_1_selection_quality_gate.csv", index=False)
print("\nQUALITY GATE")
display(quality_gate)
if (quality_gate["Status"] != "PASS").any():
    print("WARNING: One or more checks failed. Do not freeze model decisions until reviewed.")
else:
    print("All Script 14 v4 assembly checks passed. This is not proof that upstream model training is leakage-free or that any model is statistically superior.")


## Interpretation guide

- **Best observed point candidate** is the lowest-RMSE series on the development period for that task. It is a descriptive ranking, not proof of statistical superiority.
- **Random Walk** remains the point-forecast benchmark, but it is not the only candidate considered.
- **Volatility candidates** are compared against EWMA on paired forecast origins. Their target dates are mapped from the point-forecast origin/horizon table, and targets on or after 2024-01-01 are excluded.
- Different volatility candidates can have different eligible sample sizes when some source models have missing forecasts. Always inspect `14_v4_1_volatility_target_boundary_audit.csv`, the `N` column, and `Extreme_Loss_Concentration_Flag` before comparing their scores.
- Do not treat candidates flagged for extreme values or loss concentration as validated winners. Investigate their raw forecasts and sensitivity first.
- Do not use 2024–2025 holdout results to tune models or change selection rules.
